# Remote jobs that pay in USD — 5-minute starter

Loads the free weekly sample from [`lokerdollar-market-data`](https://github.com/kelvindesman/lokerdollar-market-data)
and answers three questions: who pays the most, what the pay distribution looks like,
and which skills show up in the highest-paying roles.

Nothing to install, nothing to authenticate. Runs top-to-bottom in Colab.

The sample is 50 salary-disclosing roles, refreshed weekly. The full daily dataset
(every active role, with the apply URL) lives at <https://lokerdollar.com/en/data>.

Data licensed **CC BY 4.0** — reuse freely with attribution to lokerdollar.com.


In [ ]:
import json
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

CSV = "https://raw.githubusercontent.com/kelvindesman/lokerdollar-market-data/main/data/sample-remote-jobs.csv"

df = pd.read_csv(CSV)
print(f"{len(df)} roles, {df['company'].nunique()} companies")
df.head()


## 1. Who pays the most?


In [ ]:
cols = ['title', 'company', 'remote_type', 'pay_min', 'pay_max', 'pay_currency', 'pay_period']
yearly = df[df['pay_period'] == 'yearly'].copy()
yearly.sort_values('pay_max', ascending=False)[cols].head(10)


## 2. What does the pay range actually look like?

Midpoint of the posted range, so a wide band doesn't count twice.


In [ ]:
yearly['mid'] = (yearly['pay_min'] + yearly['pay_max']) / 2

ax = yearly['mid'].plot(kind='hist', bins=12, edgecolor='white')
ax.set_xlabel('Posted salary midpoint (USD/yr)')
ax.set_ylabel('Roles')
ax.set_title('Posted pay, remote USD roles')
plt.show()

yearly['mid'].describe().round(0)


## 3. Which skills travel with the top of the range?

`skills_required` is a JSON array per row — this is the column a raw job feed does not have.


In [ ]:
def skills(series):
    out = Counter()
    for raw in series.dropna():
        try:
            out.update(json.loads(raw))
        except (ValueError, TypeError):
            continue
    return out

top_half = yearly[yearly['mid'] >= yearly['mid'].median()]

pd.DataFrame(
    skills(top_half['skills_required']).most_common(15),
    columns=['skill', 'roles'],
)


## 4. Green and red flags

Signals extracted per posting — async culture, equity, visa support on one side;
unpaid trials and vague scope on the other.


In [ ]:
green = skills(df['green_flags'])
red = skills(df['red_flags'])

print('Most common green flags:')
for name, n in green.most_common(8):
    print(f'  {n:>3}  {name}')

print()
print('Most common red flags:')
for name, n in red.most_common(8):
    print(f'  {n:>3}  {name}')


---

## Go further

- **Full dataset** (every active role + apply URL): <https://lokerdollar.com/en/data>
- **Free market reports** built on the same corpus: <https://lokerdollar.com/en/reports/roles>
- **Schema**: [`datapackage.json`](https://github.com/kelvindesman/lokerdollar-market-data/blob/main/datapackage.json) (Frictionless Data Package)

Built something with this? Open an issue on the repo — good analyses get linked from the reports.

```
Source: Loker Dollar Remote Job Market Data (https://lokerdollar.com/en/data), CC BY 4.0.
```
